# Consolidated Facility Forecast / Development Wedge Engine

CSV-only JupyterLite notebook.

**Facility_Master.csv**
- A = Facility
- B = Well Name
- C = Flag
- D = Working Interest

Flags:
- blank / PDP = normal PDP
- YES = remove forecast only in PDP-off sensitivity
- CERTAIN DEV = confirmed future development
- UNCERTAIN DEV LOW / MED / HIGH = uncertain development tiers

Scenario definitions:
- BASE = all PDPs online
- PDP_OFF = flagged PDP forecasts removed from both Modified and Plan, history kept
- CERTAIN_DEV = BASE + certain dev
- UNCERTAIN_LOW = BASE + certain + LOW
- UNCERTAIN_MED = BASE + certain + LOW + MED
- UNCERTAIN_HIGH = BASE + certain + LOW + MED + HIGH
- ABSOLUTE_LOW = PDP_OFF + certain + LOW
- ABSOLUTE_HIGH = BASE + certain + LOW + MED + HIGH

All volumes are WI-adjusted at well-month level. PDP and dev wells are capped to MAX_WELL_LIFE_MONTHS. Uncertain dev also has a calendar cutoff at 2050-12.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# =========================
# SETTINGS
# =========================
FACILITY_MASTER_FILE = "Facility_Master.csv"

MODIFIED_PRODUCTION_FILE = "ModifiedProduction.csv"
MODIFIED_FORECAST_FILE   = "ModifiedForecast.csv"
PLAN_PRODUCTION_FILE     = "PlanProduction.csv"
PLAN_FORECAST_FILE       = "PlanForecast.csv"

CERTAIN_DEV_FILE   = "Certain_Dev_Production.csv"
UNCERTAIN_DEV_FILE = "Uncertain_Dev_Production.csv"

CERTAIN_DEV_FILE_MODE   = "horizontal"   # "horizontal" or "combocurve"
UNCERTAIN_DEV_FILE_MODE = "horizontal"   # "horizontal" or "combocurve"

DEV_BLOCK_WIDTH = 7
DEV_DATA_START_ROW = 4
DEV_DATE_OFFSET = 0
DEV_OIL_OFFSET  = 2
DEV_GAS_OFFSET  = 4

MAX_WELL_LIFE_MONTHS = 600
UNCERTAIN_DEV_END_DATE = pd.Timestamp("2050-12-01")
POST_2034_START = pd.Timestamp("2034-01-01")

SAVE_PLOTS = True
PLOT_SCENARIOS = [
    "BASE","PDP_OFF","CERTAIN_DEV",
    "UNCERTAIN_LOW","UNCERTAIN_MED","UNCERTAIN_HIGH",
    "ABSOLUTE_LOW","ABSOLUTE_HIGH"
]

print("Settings loaded.")


In [ ]:

# =========================
# HELPERS
# =========================
def normalize_well_name(value):
    if pd.isna(value): return ""
    return re.sub(r"\s+", " ", str(value).strip()).upper()

def normalize_flag(value):
    if pd.isna(value): return ""
    x = re.sub(r"\s+", " ", str(value).strip().upper().replace("_"," ").replace("-"," "))
    return x

def normalize_wi(value):
    x = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(x): return np.nan
    return float(x/100.0 if x > 1 else x)

def month_start(series):
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

def safe_filename(value):
    return re.sub(r"[^A-Za-z0-9._-]+","_",str(value).strip()).strip("_") or "Facility"

def parse_dev_dates(series):
    out=[]
    for value in series:
        if pd.isna(value):
            out.append(pd.NaT); continue
        text=str(value).strip()
        if text=="" or text.lower()=="nan":
            out.append(pd.NaT); continue
        dt=pd.NaT
        try:
            c=pd.to_datetime(text, format="%b-%y", errors="raise")
            if 2000 <= c.year <= 2200: dt=c
        except: pass
        if pd.isna(dt):
            try:
                c=pd.to_datetime(text, errors="raise")
                if 2000 <= c.year <= 2200: dt=c
            except: pass
        if pd.isna(dt):
            try:
                n=float(value)
                if 30000 <= n <= 150000:
                    c=pd.Timestamp("1899-12-30")+pd.Timedelta(days=n)
                    if 2000 <= c.year <= 2200: dt=c
            except: pass
        if not pd.isna(dt):
            dt=pd.Timestamp(year=dt.year,month=dt.month,day=1)
        out.append(dt)
    return pd.Series(out,index=series.index,dtype="datetime64[ns]")


In [ ]:

# =========================
# FACILITY MASTER
# =========================
raw = pd.read_csv(FACILITY_MASTER_FILE)
if raw.shape[1] < 4:
    raise ValueError("Facility master needs A=Facility, B=Well Name, C=Flag, D=WI.")

master = raw.iloc[:,[0,1,2,3]].copy()
master.columns=["Facility","Well_Name","Flag","WI"]
master["Facility"]=master["Facility"].astype(str).str.strip()
master["Well_Name"]=master["Well_Name"].astype(str).str.strip()
master["Match_Name"]=master["Well_Name"].apply(normalize_well_name)
master["Flag_Norm"]=master["Flag"].apply(normalize_flag)
master["WI"]=master["WI"].apply(normalize_wi)

master=master[(master["Facility"]!="")&(master["Match_Name"]!="")].copy()
if master["WI"].isna().any() or ((master["WI"]<0)|(master["WI"]>1)).any():
    display(master[master["WI"].isna() | (master["WI"]<0) | (master["WI"]>1)])
    raise ValueError("Bad WI values.")

chk=master.groupby("Match_Name").agg(Facilities=("Facility","nunique"),WIs=("WI","nunique"))
if ((chk["Facilities"]>1)|(chk["WIs"]>1)).any():
    display(chk[(chk["Facilities"]>1)|(chk["WIs"]>1)])
    raise ValueError("A well maps to multiple facilities or WIs.")

master=master.drop_duplicates("Match_Name").reset_index(drop=True)
master["Is_Certain_Dev"]=master["Flag_Norm"].str.contains("CERTAIN DEV",regex=False)
master["Is_Uncertain_Dev"]=master["Flag_Norm"].str.contains("UNCERTAIN DEV",regex=False)
master["Uncertain_Level"]=""
master.loc[master["Is_Uncertain_Dev"]&master["Flag_Norm"].str.contains("LOW"),"Uncertain_Level"]="LOW"
master.loc[master["Is_Uncertain_Dev"]&master["Flag_Norm"].str.contains("MED"),"Uncertain_Level"]="MED"
master.loc[master["Is_Uncertain_Dev"]&master["Flag_Norm"].str.contains("HIGH"),"Uncertain_Level"]="HIGH"
master["Is_PDP"]=~(master["Is_Certain_Dev"]|master["Is_Uncertain_Dev"])
master["PDP_Remove_Sensitivity"]=master["Is_PDP"]&(master["Flag_Norm"]=="YES")

bad=master[master["Is_Uncertain_Dev"]&(master["Uncertain_Level"]=="")]
if len(bad):
    display(bad[["Facility","Well_Name","Flag"]])
    raise ValueError("Uncertain dev rows need LOW, MED, or HIGH.")

print("Facilities:",master["Facility"].nunique())
print("PDP:",int(master["Is_PDP"].sum()))
print("PDP removal sensitivity:",int(master["PDP_Remove_Sensitivity"].sum()))
print("Certain dev:",int(master["Is_Certain_Dev"].sum()))
print("Uncertain LOW/MED/HIGH:",
      int((master["Uncertain_Level"]=="LOW").sum()),
      int((master["Uncertain_Level"]=="MED").sum()),
      int((master["Uncertain_Level"]=="HIGH").sum()))


In [ ]:

# =========================
# COMBOCURVE PDP READER
# A=Well, H=Date, I=Oil BBL/month, J=Gas MCF/month
# =========================
def read_combocurve_csv(filename):
    raw=pd.read_csv(filename)
    if raw.shape[1] < 10:
        raise ValueError(f"{filename} must contain columns A-J.")
    d=raw.iloc[:,[0,7,8,9]].copy()
    d.columns=["Well_Name","Date","Oil_BBL","Gas_MCF"]
    d["Match_Name"]=d["Well_Name"].apply(normalize_well_name)
    d["Date"]=month_start(d["Date"])
    d["Oil_BBL"]=pd.to_numeric(d["Oil_BBL"],errors="coerce").fillna(0)
    d["Gas_MCF"]=pd.to_numeric(d["Gas_MCF"],errors="coerce").fillna(0)
    d=d[(d["Match_Name"]!="")&d["Date"].notna()]
    return d.groupby(["Match_Name","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum()

mod_prod=read_combocurve_csv(MODIFIED_PRODUCTION_FILE)
mod_fcst=read_combocurve_csv(MODIFIED_FORECAST_FILE)
plan_prod=read_combocurve_csv(PLAN_PRODUCTION_FILE)
plan_fcst=read_combocurve_csv(PLAN_FORECAST_FILE)


In [ ]:

# =========================
# BUILD PDP WELL PROFILES
# =========================
def build_pdp_case(case_name,prod_df,fcst_df,master_df,remove_flagged=False):
    rows=[]; qa=[]
    for _,r in master_df[master_df["Is_PDP"]].iterrows():
        well,key,fac,wi=r["Well_Name"],r["Match_Name"],r["Facility"],r["WI"]
        remove=bool(remove_flagged and r["PDP_Remove_Sensitivity"])
        p=prod_df[prod_df["Match_Name"]==key].sort_values("Date").copy()
        f=fcst_df[fcst_df["Match_Name"]==key].sort_values("Date").copy()
        fp,ff=len(p)>0,len(f)>0
        if fp: start=p["Date"].min()
        elif ff: start=f["Date"].min()
        else:
            qa.append({"Case":case_name,"Facility":fac,"Well_Name":well,"QA_Status":"MISSING FROM BOTH"})
            continue
        prof=pd.DataFrame({"Date":pd.date_range(start=start,periods=MAX_WELL_LIFE_MONTHS,freq="MS")})
        last_actual=p["Date"].max() if fp else pd.NaT
        p2=p[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"A_Oil","Gas_MCF":"A_Gas"})
        f2=f[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"F_Oil","Gas_MCF":"F_Gas"})
        prof=prof.merge(p2,on="Date",how="left").merge(f2,on="Date",how="left")
        if fp:
            am=prof["Date"]<=last_actual
            fm=prof["Date"]>last_actual
        else:
            am=pd.Series(False,index=prof.index); fm=pd.Series(True,index=prof.index)
        prof["Gross_Oil_BBL"]=0.0; prof["Gross_Gas_MCF"]=0.0; prof["Data_Source"]=""
        prof.loc[am,"Gross_Oil_BBL"]=prof.loc[am,"A_Oil"].fillna(0)
        prof.loc[am,"Gross_Gas_MCF"]=prof.loc[am,"A_Gas"].fillna(0)
        prof.loc[am,"Data_Source"]="Actual"
        if remove:
            prof.loc[fm,"Data_Source"]="Forecast Removed"
        else:
            prof.loc[fm,"Gross_Oil_BBL"]=prof.loc[fm,"F_Oil"].fillna(0)
            prof.loc[fm,"Gross_Gas_MCF"]=prof.loc[fm,"F_Gas"].fillna(0)
            prof.loc[fm,"Data_Source"]="Forecast"
        prof["Oil_BBL"]=prof["Gross_Oil_BBL"]*wi
        prof["Gas_MCF"]=prof["Gross_Gas_MCF"]*wi
        prof["Case"]=case_name; prof["Facility"]=fac; prof["Well_Name"]=well; prof["Match_Name"]=key; prof["WI"]=wi
        rows.append(prof[["Case","Facility","Well_Name","Match_Name","WI","Date","Data_Source","Oil_BBL","Gas_MCF"]])
        qa.append({"Case":case_name,"Facility":fac,"Well_Name":well,"WI":wi,
                   "Found_Production":fp,"Found_Forecast":ff,"Forecast_Removed":remove,
                   "Profile_Start":start,"Last_Actual_Month":last_actual,"QA_Status":"OK"})
    if not rows: raise ValueError(f"No PDP profiles built for {case_name}")
    return pd.concat(rows,ignore_index=True),pd.DataFrame(qa)

mod_pdp_base,mod_base_qa=build_pdp_case("Modified Base",mod_prod,mod_fcst,master,False)
plan_pdp_base,plan_base_qa=build_pdp_case("Plan Base",plan_prod,plan_fcst,master,False)
mod_pdp_off,mod_off_qa=build_pdp_case("Modified PDP Off",mod_prod,mod_fcst,master,True)
plan_pdp_off,plan_off_qa=build_pdp_case("Plan PDP Off",plan_prod,plan_fcst,master,True)


In [ ]:

# =========================
# DEV LOADERS
# =========================
def read_horizontal_dev_csv(filename):
    raw=pd.read_csv(filename,header=None)
    out=[]
    for start in range(0,raw.shape[1],DEV_BLOCK_WIDTH):
        if start+DEV_GAS_OFFSET>=raw.shape[1]: continue
        w=raw.iloc[0,start]
        if pd.isna(w) or str(w).strip()=="": continue
        t=pd.DataFrame({
            "Well_Name":str(w).strip(),
            "Date":raw.iloc[DEV_DATA_START_ROW:,start+DEV_DATE_OFFSET].values,
            "Oil_BBL":raw.iloc[DEV_DATA_START_ROW:,start+DEV_OIL_OFFSET].values,
            "Gas_MCF":raw.iloc[DEV_DATA_START_ROW:,start+DEV_GAS_OFFSET].values
        })
        t["Date"]=parse_dev_dates(t["Date"])
        t["Oil_BBL"]=pd.to_numeric(t["Oil_BBL"],errors="coerce").fillna(0)
        t["Gas_MCF"]=pd.to_numeric(t["Gas_MCF"],errors="coerce").fillna(0)
        t["Match_Name"]=t["Well_Name"].apply(normalize_well_name)
        t=t[t["Date"].notna()&(t["Match_Name"]!="")]
        out.append(t)
    if not out: raise ValueError(f"No dev blocks found in {filename}")
    return pd.concat(out,ignore_index=True)

def read_dev_csv(filename,mode):
    if mode.lower()=="horizontal":
        return read_horizontal_dev_csv(filename)
    if mode.lower()=="combocurve":
        d=read_combocurve_csv(filename)
        d["Well_Name"]=d["Match_Name"]
        return d
    raise ValueError("mode must be horizontal or combocurve")

def prepare_dev_profiles(dev_df,master_df,dev_type,level=None,calendar_end=None):
    if dev_type=="CERTAIN":
        m=master_df[master_df["Is_Certain_Dev"]].copy()
    else:
        m=master_df[master_df["Is_Uncertain_Dev"]&(master_df["Uncertain_Level"]==level)].copy()
    d=dev_df.merge(m[["Match_Name","Facility","Well_Name","WI"]],on="Match_Name",how="inner",suffixes=("_File","_Master"))
    if len(d)==0:
        return pd.DataFrame(columns=["Facility","Well_Name","Match_Name","WI","Date","Oil_BBL","Gas_MCF"])
    out=[]
    for key,g in d.groupby("Match_Name"):
        g=g.sort_values("Date").copy()
        start=g["Date"].min()
        cutoff=start+pd.DateOffset(months=MAX_WELL_LIFE_MONTHS-1)
        if calendar_end is not None:
            cutoff=min(cutoff,pd.Timestamp(calendar_end))
        g=g[(g["Date"]>=start)&(g["Date"]<=cutoff)]
        g=g.groupby(["Match_Name","Facility","Well_Name_Master","WI","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum()
        g["Oil_BBL"]=g["Oil_BBL"]*g["WI"]
        g["Gas_MCF"]=g["Gas_MCF"]*g["WI"]
        g["Well_Name"]=g["Well_Name_Master"]
        out.append(g[["Facility","Well_Name","Match_Name","WI","Date","Oil_BBL","Gas_MCF"]])
    return pd.concat(out,ignore_index=True)

certain_raw=read_dev_csv(CERTAIN_DEV_FILE,CERTAIN_DEV_FILE_MODE)
uncertain_raw=read_dev_csv(UNCERTAIN_DEV_FILE,UNCERTAIN_DEV_FILE_MODE)

certain_dev=prepare_dev_profiles(certain_raw,master,"CERTAIN")
unc_low=prepare_dev_profiles(uncertain_raw,master,"UNCERTAIN","LOW",UNCERTAIN_DEV_END_DATE)
unc_med=prepare_dev_profiles(uncertain_raw,master,"UNCERTAIN","MED",UNCERTAIN_DEV_END_DATE)
unc_high=prepare_dev_profiles(uncertain_raw,master,"UNCERTAIN","HIGH",UNCERTAIN_DEV_END_DATE)


In [ ]:

# =========================
# FACILITY AGGREGATION + SCENARIOS
# =========================
def aggregate_component(df):
    if df is None or len(df)==0:
        return pd.DataFrame(columns=["Facility","Date","Oil_BBL","Gas_MCF"])
    return df.groupby(["Facility","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum().sort_values(["Facility","Date"])

def combine_components(*xs):
    xs=[x for x in xs if x is not None and len(x)>0]
    if not xs:
        return pd.DataFrame(columns=["Facility","Date","Oil_BBL","Gas_MCF"])
    return pd.concat(xs,ignore_index=True).groupby(["Facility","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum().sort_values(["Facility","Date"])

mod_base=aggregate_component(mod_pdp_base); plan_base=aggregate_component(plan_pdp_base)
mod_off=aggregate_component(mod_pdp_off); plan_off=aggregate_component(plan_pdp_off)
certain=aggregate_component(certain_dev)
low=aggregate_component(unc_low); med=aggregate_component(unc_med); high=aggregate_component(unc_high)

scenario_components={
"BASE":{"Modified":mod_base,"Plan":plan_base},
"PDP_OFF":{"Modified":mod_off,"Plan":plan_off},
"CERTAIN_DEV":{"Modified":combine_components(mod_base,certain),"Plan":combine_components(plan_base,certain)},
"UNCERTAIN_LOW":{"Modified":combine_components(mod_base,certain,low),"Plan":combine_components(plan_base,certain,low)},
"UNCERTAIN_MED":{"Modified":combine_components(mod_base,certain,low,med),"Plan":combine_components(plan_base,certain,low,med)},
"UNCERTAIN_HIGH":{"Modified":combine_components(mod_base,certain,low,med,high),"Plan":combine_components(plan_base,certain,low,med,high)},
"ABSOLUTE_LOW":{"Modified":combine_components(mod_off,certain,low),"Plan":combine_components(plan_off,certain,low)},
"ABSOLUTE_HIGH":{"Modified":combine_components(mod_base,certain,low,med,high),"Plan":combine_components(plan_base,certain,low,med,high)}
}


In [ ]:

# =========================
# MONTHLY MODIFIED VS PLAN TABLES
# =========================
def build_comparison(name,mod,plan):
    c=pd.merge(mod,plan,on=["Facility","Date"],how="outer",suffixes=("_Modified","_Plan"))
    c=c.sort_values(["Facility","Date"]).reset_index(drop=True)
    for col in ["Oil_BBL_Modified","Gas_MCF_Modified","Oil_BBL_Plan","Gas_MCF_Plan"]:
        c[col]=pd.to_numeric(c[col],errors="coerce").fillna(0)
    c["Days_In_Month"]=c["Date"].dt.days_in_month
    c["Oil_BPD_Modified"]=c["Oil_BBL_Modified"]/c["Days_In_Month"]
    c["Oil_BPD_Plan"]=c["Oil_BBL_Plan"]/c["Days_In_Month"]
    c["Gas_MCFD_Modified"]=c["Gas_MCF_Modified"]/c["Days_In_Month"]
    c["Gas_MCFD_Plan"]=c["Gas_MCF_Plan"]/c["Days_In_Month"]
    c["Cum_Oil_BBL_Modified"]=c.groupby("Facility")["Oil_BBL_Modified"].cumsum()
    c["Cum_Oil_BBL_Plan"]=c.groupby("Facility")["Oil_BBL_Plan"].cumsum()
    c["Cum_Gas_MCF_Modified"]=c.groupby("Facility")["Gas_MCF_Modified"].cumsum()
    c["Cum_Gas_MCF_Plan"]=c.groupby("Facility")["Gas_MCF_Plan"].cumsum()
    c["Oil_Cum_Delta_BBL"]=c["Cum_Oil_BBL_Modified"]-c["Cum_Oil_BBL_Plan"]
    c["Gas_Cum_Delta_MCF"]=c["Cum_Gas_MCF_Modified"]-c["Cum_Gas_MCF_Plan"]
    c.insert(0,"Scenario",name)
    return c

scenario_tables={k:build_comparison(k,v["Modified"],v["Plan"]) for k,v in scenario_components.items()}
monthly_all=pd.concat(scenario_tables.values(),ignore_index=True)
display(monthly_all.head())


In [ ]:

# =========================
# SUMMARY CSV
# overall peaks, post-2034 peaks, total & post-2034 cumulatives
# =========================
def peak_and_date(d,col):
    if len(d)==0:return np.nan,pd.NaT
    i=d[col].idxmax()
    return d.loc[i,col],d.loc[i,"Date"]

def summarize(df):
    rows=[]
    for fac,d in df.groupby("Facility"):
        d=d.sort_values("Date").copy()
        post=d[d["Date"]>=POST_2034_START].copy()
        f=d.iloc[-1]
        opm,opmd=peak_and_date(d,"Oil_BPD_Modified"); opp,oppd=peak_and_date(d,"Oil_BPD_Plan")
        gpm,gpmd=peak_and_date(d,"Gas_MCFD_Modified"); gpp,gppd=peak_and_date(d,"Gas_MCFD_Plan")
        pom,pomd=peak_and_date(post,"Oil_BPD_Modified"); pop,popd=peak_and_date(post,"Oil_BPD_Plan")
        pgm,pgmd=peak_and_date(post,"Gas_MCFD_Modified"); pgp,pgpd=peak_and_date(post,"Gas_MCFD_Plan")
        oil_delta=f["Cum_Oil_BBL_Modified"]-f["Cum_Oil_BBL_Plan"]
        gas_delta=f["Cum_Gas_MCF_Modified"]-f["Cum_Gas_MCF_Plan"]
        rows.append({
            "Scenario":f["Scenario"],"Facility":fac,
            "Final_Oil_Cum_BBL_Modified":f["Cum_Oil_BBL_Modified"],
            "Final_Oil_Cum_BBL_Plan":f["Cum_Oil_BBL_Plan"],
            "Final_Oil_Cum_Delta_BBL":oil_delta,
            "Final_Oil_Cum_Delta_Pct_vs_Plan":oil_delta/f["Cum_Oil_BBL_Plan"]*100 if f["Cum_Oil_BBL_Plan"] else np.nan,
            "Final_Gas_Cum_MCF_Modified":f["Cum_Gas_MCF_Modified"],
            "Final_Gas_Cum_MCF_Plan":f["Cum_Gas_MCF_Plan"],
            "Final_Gas_Cum_Delta_MCF":gas_delta,
            "Final_Gas_Cum_Delta_Pct_vs_Plan":gas_delta/f["Cum_Gas_MCF_Plan"]*100 if f["Cum_Gas_MCF_Plan"] else np.nan,
            "Overall_Peak_Oil_BPD_Modified":opm,"Overall_Peak_Oil_Date_Modified":opmd,
            "Overall_Peak_Oil_BPD_Plan":opp,"Overall_Peak_Oil_Date_Plan":oppd,
            "Overall_Peak_Gas_MCFD_Modified":gpm,"Overall_Peak_Gas_Date_Modified":gpmd,
            "Overall_Peak_Gas_MCFD_Plan":gpp,"Overall_Peak_Gas_Date_Plan":gppd,
            "Post2034_Peak_Oil_BPD_Modified":pom,"Post2034_Peak_Oil_Date_Modified":pomd,
            "Post2034_Peak_Oil_BPD_Plan":pop,"Post2034_Peak_Oil_Date_Plan":popd,
            "Post2034_Peak_Gas_MCFD_Modified":pgm,"Post2034_Peak_Gas_Date_Modified":pgmd,
            "Post2034_Peak_Gas_MCFD_Plan":pgp,"Post2034_Peak_Gas_Date_Plan":pgpd,
            "Post2034_Oil_Cum_BBL_Modified":post["Oil_BBL_Modified"].sum(),
            "Post2034_Oil_Cum_BBL_Plan":post["Oil_BBL_Plan"].sum(),
            "Post2034_Gas_Cum_MCF_Modified":post["Gas_MCF_Modified"].sum(),
            "Post2034_Gas_Cum_MCF_Plan":post["Gas_MCF_Plan"].sum()
        })
    return pd.DataFrame(rows)

summary_all=pd.concat([summarize(x) for x in scenario_tables.values()],ignore_index=True)
display(summary_all)


In [ ]:

# =========================
# RATE + CUMULATIVE PNGS
# =========================
def plot_rate_cum(df,facility,scenario,stream="Oil"):
    d=df[df["Facility"]==facility].sort_values("Date").copy()
    if len(d)==0:return
    if stream=="Oil":
        rm,rp="Oil_BPD_Modified","Oil_BPD_Plan"
        cm,cp="Cum_Oil_BBL_Modified","Cum_Oil_BBL_Plan"
        runit,cunit="BPD","BBL"
    else:
        rm,rp="Gas_MCFD_Modified","Gas_MCFD_Plan"
        cm,cp="Cum_Gas_MCF_Modified","Cum_Gas_MCF_Plan"
        runit,cunit="MCF/D","MCF"
    x=list(d["Date"].dt.to_pydatetime())
    rmod=d[rm].to_numpy(float); rplan=d[rp].to_numpy(float)
    cmod=d[cm].to_numpy(float); cplan=d[cp].to_numpy(float)
    pmod,pplan=np.max(rmod),np.max(rplan)
    fmod,fplan=cmod[-1],cplan[-1]
    delta=fmod-fplan
    pct=delta/fplan*100 if fplan else np.nan
    fig,ax1=plt.subplots(figsize=(14,7)); ax2=ax1.twinx()
    l1,=ax1.plot(x,rmod,linewidth=2,label="Modified Rate")
    l2,=ax1.plot(x,rplan,linewidth=2,label="Plan Rate")
    l3,=ax2.plot(x,cmod,"--",linewidth=2,label="Modified Cumulative")
    l4,=ax2.plot(x,cplan,"--",linewidth=2,label="Plan Cumulative")
    ax2.fill_between(x,cmod,cplan,color="red",alpha=0.15)
    ax1.text(0.015,0.97,
             f"Peak Modified: {pmod:,.0f} {runit}\nPeak Plan: {pplan:,.0f} {runit}\n"
             f"Final Modified Cum: {fmod:,.0f} {cunit}\nFinal Plan Cum: {fplan:,.0f} {cunit}",
             transform=ax1.transAxes,va="top",
             bbox=dict(boxstyle="round",facecolor="white",alpha=0.88))
    ax2.text(0.985,0.92,
             f"End Cum Delta: {delta:+,.0f} {cunit}\nDelta vs Plan: {pct:+.2f}%" if np.isfinite(pct)
             else f"End Cum Delta: {delta:+,.0f} {cunit}\nDelta vs Plan: N/A",
             transform=ax2.transAxes,va="top",ha="right",
             bbox=dict(boxstyle="round",facecolor="white",alpha=0.88))
    ax1.set_title(f"{facility} - {scenario} - {stream} Rate + Cumulative: Modified vs Plan")
    ax1.set_xlabel("Date"); ax1.set_ylabel(f"{stream} Rate ({runit})"); ax2.set_ylabel(f"Cumulative {stream} ({cunit})")
    ax1.grid(True,alpha=0.3)
    ax1.legend([l1,l2,l3,l4],["Modified Rate","Plan Rate","Modified Cumulative","Plan Cumulative"],loc="lower right")
    if SAVE_PLOTS:
        fig.savefig(f"{safe_filename(facility)}_{safe_filename(scenario)}_{stream}.png",dpi=160,bbox_inches="tight")
    plt.show()

if SAVE_PLOTS:
    for scen in PLOT_SCENARIOS:
        if scen not in scenario_tables: continue
        for fac in sorted(scenario_tables[scen]["Facility"].dropna().unique()):
            plot_rate_cum(scenario_tables[scen],fac,scen,"Oil")
            plot_rate_cum(scenario_tables[scen],fac,scen,"Gas")


In [ ]:

# =========================
# SAVE OUTPUTS
# =========================
monthly_all.to_csv("Facility_Monthly_All_Scenarios.csv",index=False)
summary_all.to_csv("Facility_Summary_All_Scenarios.csv",index=False)

monthly_all[monthly_all["Scenario"]=="ABSOLUTE_LOW"].to_csv("Facility_Monthly_Absolute_Low.csv",index=False)
monthly_all[monthly_all["Scenario"]=="ABSOLUTE_HIGH"].to_csv("Facility_Monthly_Absolute_High.csv",index=False)
summary_all[summary_all["Scenario"]=="ABSOLUTE_LOW"].to_csv("Facility_Summary_Absolute_Low.csv",index=False)
summary_all[summary_all["Scenario"]=="ABSOLUTE_HIGH"].to_csv("Facility_Summary_Absolute_High.csv",index=False)

master.to_csv("Facility_Master_Cleaned_QA.csv",index=False)
pd.concat([mod_base_qa,plan_base_qa,mod_off_qa,plan_off_qa],ignore_index=True).to_csv("PDP_Profile_QA.csv",index=False)

print("Done.")
